In [0]:
# Spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ML utilities
import numpy as np
import pandas as pd

# XGBoost
from xgboost import XGBClassifier

# Sklearn helpers
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"

df = spark.table(MASTER_PATH)

In [0]:
display(df)

In [0]:
print(df.columns)

In [0]:
stint_window = (
    Window
    .partitionBy("Driver", "Year", "Circuit", "Stint")
    .orderBy("LapNumber")
)

df = df.withColumn(
    "lap_delta",
    F.col("LapTime") - F.lag("LapTime").over(stint_window)
)

df = df.withColumn(
    "lap_delta",
    F.when((F.col("lap_delta") < -5) | (F.col("lap_delta") > 5), None)
     .otherwise(F.col("lap_delta"))
)

In [0]:
# 1. Race normalized lap pace
race_avg = df.groupBy("Year","Circuit","LapNumber") \
             .agg(F.avg("LapTime").alias("race_avg_laptime"))

df = df.join(race_avg, ["Year","Circuit","LapNumber"])

df = df.withColumn(
    "relative_laptime",
    F.col("LapTime") - F.col("race_avg_laptime")
)

# 2. Sector normalization
sector_avg = df.groupBy("Year","Circuit").agg(
    F.avg("Sector1Time").alias("sector1_avg"),
    F.avg("Sector2Time").alias("sector2_avg"),
    F.avg("Sector3Time").alias("sector3_avg")
)

df = df.join(sector_avg, ["Year","Circuit"])

df = df.withColumn("sector1_rel", F.col("Sector1Time") - F.col("sector1_avg"))
df = df.withColumn("sector2_rel", F.col("Sector2Time") - F.col("sector2_avg"))
df = df.withColumn("sector3_rel", F.col("Sector3Time") - F.col("sector3_avg"))

# 3. Overtaking ability
df = df.withColumn(
    "position_gain",
    F.col("QualiPosition") - F.col("FinalPosition")
)

# 4. Tyre management
stint_life = df.groupBy("Driver","Year","Stint") \
               .agg(F.max("TyreLife").alias("stint_life"))

tyre_management = stint_life.groupBy("Driver") \
                            .agg(F.avg("stint_life").alias("tyre_management"))

# 5. Driver consistency
consistency = df.groupBy("Driver") \
                .agg(F.stddev("LapTime").alias("lap_consistency"))

# 6. Driver profile
driver_profile = df.groupBy("Driver").agg(
    F.avg("relative_laptime").alias("driver_pace"),
    F.avg("sector1_rel").alias("driver_sector1_skill"),
    F.avg("sector2_rel").alias("driver_sector2_skill"),
    F.avg("sector3_rel").alias("driver_sector3_skill"),
    F.avg("position_gain").alias("driver_overtake_skill"),
    F.avg("SpeedI1").alias("driver_speedI1"),
    F.avg("SpeedI2").alias("driver_speedI2"),
    F.avg("SpeedFL").alias("driver_speedFL"),
    F.avg("SpeedST").alias("driver_speedST")
)

driver_profile = driver_profile.join(consistency, "Driver")
driver_profile = driver_profile.join(tyre_management, "Driver")

print(driver_profile.columns)

In [0]:
# 8. Merge driver profile into main dataframe
df = df.join(driver_profile, "Driver", "left")

display(df)

### **Step 2: Target Construction**

In [0]:
finish_window = Window.partitionBy("Year", "Circuit", "Driver")

df = df.withColumn(
    "FinalPosition",
    F.max("Position").over(finish_window)
)

In [0]:
df = (
    df
    .withColumn("PodiumFinish", (F.col("FinalPosition") <= 3).cast("int"))
    .withColumn("WinFinish", (F.col("FinalPosition") == 1).cast("int"))
)

In [0]:
weather_window = Window.partitionBy("Year", "Circuit")

df = (
    df
    .withColumn("AirTemp_delta", F.col("AirTemp_C") - F.avg("AirTemp_C").over(weather_window))
    .withColumn("TrackTemp_delta", F.col("TrackTemp_C") - F.avg("TrackTemp_C").over(weather_window))
    .withColumn("Humidity_delta", F.col("Humidity_pct") - F.avg("Humidity_pct").over(weather_window))
    .withColumn("WindSpeed_delta", F.col("WindSpeed_kmh") - F.avg("WindSpeed_kmh").over(weather_window))
)

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.default.f1_ml_lap_dataset")

### **Step 3: Feature Selection**

In [0]:
FEATURES_PRERACE = [
    "Driver",
    "TeamName",
    "Circuit",
    "Year",
    "QualiPosition",

    # Driver style
    'driver_pace', 
    'driver_sector1_skill',
    'driver_sector2_skill',
    'driver_sector3_skill',
    'driver_overtake_skill',
    'driver_speedI1',
    'driver_speedI2',
    'driver_speedFL',
    'driver_speedST',
    'lap_consistency',
    'tyre_management',
]

In [0]:
FEATURES_LIVE = [
    "LapNumber",
    "RacePhase",
    "Position",
    "GapToAhead",
    "DeltaToLeader",

    "Compound",
    "TyreLife",
    "Stint",
    "TrackStatus",

    # Driver style (allowed)
    'driver_pace', 
    'driver_sector1_skill',
    'driver_sector2_skill',
    'driver_sector3_skill',
    'driver_overtake_skill',
    'driver_speedI1',
    'driver_speedI2',
    'driver_speedFL',
    'driver_speedST',
    'lap_consistency',
    'tyre_management',

    # Weather (relative only)
    "AirTemp_delta",
    "TrackTemp_delta",
    "Humidity_delta",
    "WindSpeed_delta"
]


### **Step 4: Convert to Pandas**

In [0]:
df_prerace = df.select(FEATURES_PRERACE + ["PodiumFinish"])
df_live = df.select(FEATURES_LIVE + ["PodiumFinish"])

pdf_prerace = df_prerace.toPandas()
pdf_live = df_live.toPandas()

In [0]:
CATEGORICAL_PRERACE = ["Driver", "TeamName", "Circuit"]
CATEGORICAL_LIVE = ["Compound", "RacePhase"]

for c in CATEGORICAL_PRERACE:
    pdf_prerace[c] = pdf_prerace[c].astype("category")

for c in CATEGORICAL_LIVE:
    pdf_live[c] = pdf_live[c].astype("category")

### **Step 6: Train Pre-Race Strategy Probability Model**

In [0]:
# Sort by year just to be safe
pdf_prerace = pdf_prerace.sort_values("Year")

X_pre = pdf_prerace[FEATURES_PRERACE]
y_pre = pdf_prerace["PodiumFinish"]

# ---------# Time-based split
# ---------
TRAIN_END_YEAR = 2020
VAL_YEAR = 2021
TEST_START_YEAR = 2022

# Train
train_mask = pdf_prerace["Year"] <= TRAIN_END_YEAR

# Validation (optional but recommended for tuning)
val_mask = pdf_prerace["Year"] == VAL_YEAR

# Test (future simulation)
test_mask = pdf_prerace["Year"] >= TEST_START_YEAR

X_pre_train = X_pre.loc[train_mask].copy()
y_pre_train = y_pre.loc[train_mask].copy()

X_pre_val = X_pre.loc[val_mask].copy()
y_pre_val = y_pre.loc[val_mask].copy()

X_pre_test = X_pre.loc[test_mask].copy()
y_pre_test = y_pre.loc[test_mask].copy()

In [0]:
pdf_prerace.head()

In [0]:
pre_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist"
)

pre_model.fit(X_pre_train, y_pre_train)


In [0]:
from sklearn.metrics import roc_auc_score, classification_report

probs = pre_model.predict_proba(X_pre_test)[:, 1]
auc = roc_auc_score(y_pre_test, probs)

print("Pre-race AUC (unseen races):", round(auc, 4))
#print(classification_report(y_test, pre_model.predict(X_test)))

### **Step 5: Train Live Podium Probability Model**

In [0]:
groups = pdf_live.index  # lap-wise independence

gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(pdf_live, groups=groups))

X_train = pdf_live.iloc[train_idx][FEATURES_LIVE]
y_train = pdf_live.iloc[train_idx]["PodiumFinish"]

X_test = pdf_live.iloc[test_idx][FEATURES_LIVE]
y_test = pdf_live.iloc[test_idx]["PodiumFinish"]

live_model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist"
)

live_model.fit(X_train, y_train)


In [0]:
probs = live_model.predict_proba(X_test)[:, 1]
print("LIVE MODEL AUC:", roc_auc_score(y_test, probs))

In [0]:
baseline_podium = pre_model.predict_proba(X_pre.iloc[[0]])[0][1]
live_podium = live_model.predict_proba(X_test.iloc[[0]])[0][1]

delta = live_podium - baseline_podium
print(delta)

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(pre_model, max_num_features=15)
plt.show()

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(live_model, max_num_features=15)
plt.show()

In [0]:
probs = pre_model.predict_proba(X_pre)[:, 1]
roc_auc_score(y_pre, probs)

In [0]:
from sklearn.calibration import calibration_curve

# predicted probabilities
probs = pre_model.predict_proba(X_pre)[:, 1]

# calibration data
prob_true, prob_pred = calibration_curve(
    y_pre,
    probs,
    n_bins=10,
    strategy="uniform"
)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))

# Perfect calibration reference
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

# Your model
plt.plot(prob_pred, prob_true, marker="o", label="Pre-race model")

plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration curve (Pre-race model)")
plt.legend()
plt.grid(True)

plt.show()

In [0]:
live_probs = live_model.predict_proba(X_test)[:, 1]

prob_true_l, prob_pred_l = calibration_curve(
    y_test,
    live_probs,
    n_bins=10
)

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--")
plt.plot(prob_pred_l, prob_true_l, marker="o")
plt.title("Calibration curve (Live model)")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.grid(True)
plt.show()


In [0]:
# Save Model to local filesystem only (serverless clusters cannot write to /dbfs/FileStore)
pre_model.save_model("../models/pre_model.json")
live_model.save_model("../models/live_model.json")

# Note: Copying to /dbfs/FileStore/models/ is not supported on serverless clusters. Models are saved in /models and can be downloaded from there if needed.

In [0]:
### Trial


# Convert driver profile to pandas lookup table
driver_profile_pdf = driver_profile.toPandas().set_index("Driver")

def predict_prerace(driver, team, circuit, year, quali_pos):

    # Lookup driver profile
    if driver in driver_profile_pdf.index:
        profile = driver_profile_pdf.loc[driver]
    else:
        # fallback for new driver
        profile = driver_profile_pdf.mean()

    row = pd.DataFrame([{
        "Driver": driver,
        "TeamName": team,
        "Circuit": circuit,
        "Year": year,
        "QualiPosition": quali_pos,
        "avg_deg_rate": profile["avg_deg_rate"],
        "deg_variance": profile["deg_variance"],
        "avg_stint_length": profile["avg_stint_length"],
        "pit_frequency": profile["pit_frequency"],
        "career_podium_rate": profile["career_podium_rate"]
    }])

    for c in CATEGORICAL_PRERACE:
        row[c] = row[c].astype("category")

    prob = pre_model.predict_proba(row)[0][1]
    return prob

In [0]:
l = list(pdf_prerace["TeamName"].unique())
print(l)

In [0]:
l = list(pdf_prerace["Circuit"].unique())
print(l)

In [0]:
probability = predict_prerace(
    driver="HAM",
    team="Mercedes",
    circuit="Silverstone",
    year=2026,
    quali_pos=1
) * 100

print(f"{probability:.2f}%")

In [0]:
for driver in pdf_prerace["Driver"].unique():
    probability = predict_prerace(
        driver=driver,
        team="Red Bull Racing",
        circuit="Nürburgring",
        year=2020,
        quali_pos=5
    ) * 100

    print(f"{driver} - {probability:.2f}%")